# Phase 11 — Testing PySpark Experiments Notebook

This is the **STARTER notebook** for Phase 11.

Run it **top-to-bottom**. The deterministic setup, experiment order, testing questions, and applied-project structure are preserved from the SOLUTION notebook.

Worked implementation and answer-revealing assertions have been removed so you can build the tests yourself.

Use this workflow:

```text
state the contract
    ↓
choose the smallest deterministic fixture
    ↓
classify the test boundary
    ↓
predict the correct output
    ↓
call production logic
    ↓
assert schema / identity / grain / values
    ↓
reconcile where applicable
    ↓
introduce edge cases
    ↓
prove a regression is detected
```

Core question:

> **Can the test suite detect a meaningful pipeline defect before downstream data is silently corrupted?**

Important:

- Use **PySpark 4.2.0**.
- Use single-quoted Python strings.
- Include concise inline teaching comments.
- Reuse existing Phase 9/10 logic rather than copying production logic into tests.
- Prefer small deterministic fixtures.
- Do not assume DataFrame row order.
- Do not mark Phase 11 complete from this notebook alone.


<a id="toc"></a>
## Table of Contents

- [Setup and Existing Subjects Under Test](#setup)
- [Testing Exercise Protocol](#protocol)
- [Experiment 1 — Reusable Spark Fixture](#experiment-1)
- [Experiment 2 — Unit Test a Filter Transformation](#experiment-2)
- [Experiment 3 — Deterministic DataFrame Comparison](#experiment-3)
- [Experiment 4 — Schema Testing](#experiment-4)
- [Experiment 5 — Row-Count vs. Identity Assertions](#experiment-5)
- [Experiment 6 — Grain and Primary-Key Testing](#experiment-6)
- [Experiment 7 — Composite-Key Testing](#experiment-7)
- [Experiment 8 — Referential-Integrity Testing](#experiment-8)
- [Experiment 9 — Numerical Reconciliation](#experiment-9)
- [Experiment 10 — Accepted/Rejected Quality Testing](#experiment-10)
- [Experiment 11 — Deterministic Window Testing](#experiment-11)
- [Experiment 12 — Edge Cases](#experiment-12)
- [Experiment 13 — Integration Testing](#experiment-13)
- [Experiment 14 — Distributed Spark Testing Cost](#experiment-14)
- [Experiment 15 — Regression Detection](#experiment-15)
- [Applied Phase 11 Project](#applied-project)
- [Cleanup](#cleanup)


<a id="setup"></a>
# Setup and Existing Subjects Under Test

Phase 11 protects logic already created in Phases 9 and 10.

```text
Phase 9
→ modular transformation architecture

Phase 10
→ explicit validation contracts

Phase 11
→ automated regression protection
```

[Back to Table of Contents](#toc)


In [ ]:
from datetime import date
from decimal import Decimal
import importlib.util
from pathlib import Path
from types import ModuleType

from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from pyspark.sql.types import StringType
from pyspark.sql.types import StructField
from pyspark.sql.types import StructType
from pyspark.testing.utils import assertDataFrameEqual
from pyspark.testing.utils import assertSchemaEqual


In [ ]:
spark = (
    SparkSession.builder
    .appName('phase_11_testing_pyspark_notebook')
    .master('local[2]')
    # Keep tiny test shuffles predictable and inexpensive.
    .config('spark.sql.shuffle.partitions', '2')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')


In [ ]:
def load_module_from_path(
    module_name: str,
    module_path: Path,
) -> ModuleType:
    '''Load one existing phase lecture module from its repository path.'''

    # This setup helper is provided so the exercises can call existing
    # Phase 9/10 logic instead of duplicating production behavior.
    spec = importlib.util.spec_from_file_location(
        module_name,
        module_path,
    )

    if spec is None or spec.loader is None:
        raise ImportError(f'Unable to load module from {module_path}.')

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    return module


In [ ]:
# Resolve the repository root from the current working directory.
repo_root = Path.cwd()

if repo_root.name == 'exercises':
    repo_root = repo_root.parents[2]
elif repo_root.name == 'phase_11_testing_pyspark':
    repo_root = repo_root.parents[1]

phase_09 = load_module_from_path(
    'phase_09_lecture_for_testing',
    repo_root
    / 'phases'
    / 'phase_09_pyspark_application_architecture'
    / 'phase_09_lecture.py',
)

phase_10 = load_module_from_path(
    'phase_10_lecture_for_testing',
    repo_root
    / 'phases'
    / 'phase_10_data_quality_schema_enforcement'
    / 'phase_10_lecture.py',
)


In [ ]:
# TODO:
# Implement reusable assertion helpers as you encounter the relevant experiments.
#
# Suggested helpers:
#
# def assert_unique_key(df: DataFrame, key_columns: list[str]) -> None:
#     ...
#
# def assert_no_orphans(
#     child_df: DataFrame,
#     parent_df: DataFrame,
#     key_columns: list[str],
# ) -> None:
#     ...
#
# def assert_measure_reconciles(
#     input_df: DataFrame,
#     output_df: DataFrame,
#     column_name: str,
# ) -> None:
#     ...


<a id="protocol"></a>
# Testing Exercise Protocol

For every experiment:

```text
1. State the contract.
2. Identify unit vs. integration scope.
3. State input and output grain.
4. Use the smallest deterministic fixture.
5. Predict the result before execution.
6. Call real production logic.
7. Assert business/data correctness.
8. Explain Spark actions triggered by the assertion.
9. Identify the defect the test protects against.
```

[Back to Table of Contents](#toc)


<a id="experiment-1"></a>
# Experiment 1 — Reusable Spark Fixture

Create the reusable `pytest` Spark fixture that should eventually live in `conftest.py`.

Requirements:

```text
@pytest.fixture(scope='session')
local[2]
small shuffle partition count
WARN logging
yield
clean shutdown
```

Questions:

1. Why use session scope for the Spark runtime?
2. Why should scenario-specific DataFrames usually remain function-scoped?
3. What mutable Spark session state could leak between tests?

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Write the reusable pytest Spark fixture as a string or code example.
#
# It should:
# - create one local SparkSession;
# - use local[2];
# - use a small shuffle partition count;
# - set WARN logging;
# - yield the session;
# - stop it after the test session.


<a id="experiment-2"></a>
# Experiment 2 — Unit Test a Filter Transformation

Contract:

```text
filter_orders()
→ order_status must be included
→ net_sales must meet the configured minimum
→ surviving order grain remains one row per order_id
```

Fixture:

```text
O001 | COMPLETED | 125.00
O002 | CANCELLED |  80.00
O003 | COMPLETED |  -1.00
```

Configuration:

```text
included_statuses = COMPLETED
minimum_net_sales = 0.00
```

Before running, predict the exact surviving business key(s).

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Build the three-row deterministic fixture with an explicit schema.
# 2. Call phase_09.filter_orders().
# 3. Assert the exact surviving order_id set.
# 4. Assert order_id remains unique.
#
# Do NOT stop at a row-count assertion.


<a id="experiment-3"></a>
# Experiment 3 — Deterministic DataFrame Comparison

Create two DataFrames with the same logical rows presented in different orders.

Demonstrate two safe comparison strategies:

```text
1. deterministic orderBy() for tiny assertion data;
2. assertDataFrameEqual(..., checkRowOrder=False).
```

Explain why raw unordered `collect()` equality is fragile.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Build actual_df and expected_df with identical logical rows in different orders.
#
# Then:
# - compare after deterministic ordering;
# - compare with assertDataFrameEqual(..., checkRowOrder=False).
#
# Explain why production data should not be globally sorted merely for tests.


<a id="experiment-4"></a>
# Experiment 4 — Schema Testing

Subject:

```text
phase_09.add_processing_date()
```

Protect separately:

```text
processing_date exists
processing_date type = date
processing_date value = injected run_date
```

Also demonstrate `assertSchemaEqual()`.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Build a tiny source DataFrame.
# 2. Call phase_09.add_processing_date(...).
# 3. Assert processing_date has Spark type 'date'.
# 4. Assert its value equals the injected run date.
# 5. Demonstrate assertSchemaEqual().


<a id="experiment-5"></a>
# Experiment 5 — Row-Count vs. Identity Assertions

Construct two logically different DataFrames that both have:

```text
count = 2
```

Show that a count-only assertion can pass for both, then add a business-key identity assertion that distinguishes them.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Build two DataFrames with the same row count but different order_id values.
#
# Prove:
# - count alone passes for both;
# - key identity exposes the logical difference.


<a id="experiment-6"></a>
# Experiment 6 — Grain and Primary-Key Testing

Declared grain:

```text
orders_df
= one row per order_id
```

Test:

```text
all unique keys
duplicate identical rows
duplicate conflicting rows
```

Use the real Phase 10 duplicate-key helper where appropriate.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Build an order-grain DataFrame with unique order_id values.
# 2. Prove uniqueness.
# 3. Build a duplicate-key fixture.
# 4. Use phase_10.find_duplicate_keys().
# 5. Assert the exact duplicated key.
#
# Explain why exact duplicate rows and duplicate business keys differ.


<a id="experiment-7"></a>
# Experiment 7 — Composite-Key Testing

Inventory grain:

```text
snapshot_date + store_id + product_id
```

Fixture:

```text
2026-09-05 | S001 | P001 | 10
2026-09-05 | S001 | P001 | 12
2026-09-05 | S001 | P002 |  7
```

Assert that both conflicting `P001` rows are diagnosed while `P002` remains valid.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Build the inventory fixture with phase_10.INVENTORY_SCHEMA.
# 2. Run phase_10.add_inventory_reasons().
# 3. Assert both duplicated composite-key rows contain
#    'DUPLICATE_INVENTORY_KEY'.
# 4. Confirm the nonduplicated composite key remains valid.


<a id="experiment-8"></a>
# Experiment 8 — Referential-Integrity Testing

Relationship:

```text
orders.customer_id
    →
customers.customer_id
```

Fixture:

```text
orders:
O001 | C001
O002 | C999

customers:
C001
```

Protect:

```text
O001 accepted
O002 rejected
O002 contains ORPHAN_CUSTOMER_ID
accepted orders contain no orphan customer_id
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Build the orders/customers fixture.
# 2. Run phase_10.validate_orders().
# 3. Split accepted/rejected.
# 4. Assert O001 is accepted.
# 5. Assert O002 is rejected with ORPHAN_CUSTOMER_ID.
# 6. Write/use a left-anti assertion proving accepted data has no orphans.
#
# Extension:
# Duplicate C001 in the parent dataset and explain why parent uniqueness is a
# separate contract from referential integrity.


<a id="experiment-9"></a>
# Experiment 9 — Numerical Reconciliation

Use:

```text
ON | 125.00
ON |  75.00
BC | 200.00
```

Run `build_sales_by_province()`.

Protect:

```text
exact province rows
exact order_count values
exact net_sales values
one row per province
sum(input net_sales) = sum(output net_sales)
```

Then add a per-row formula test:

```text
gross_margin = gross_sales - gross_cost
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Build the province-sales fixture.
# 2. Run phase_09.build_sales_by_province().
# 3. Assert exact groups and measures.
# 4. Assert province grain.
# 5. Reconcile total net_sales.
#
# Then:
# 6. Build a tiny fact_sales DataFrame.
# 7. Assert no row violates gross_margin = gross_sales - gross_cost.


<a id="experiment-10"></a>
# Experiment 10 — Accepted/Rejected Quality Testing

Create:

```text
valid row
row with missing customer_id + negative net_sales
orphan row
```

Protect both sides of the quality split.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Build orders and customers fixtures.
# 2. Run phase_10.validate_orders().
# 3. Split accepted/rejected.
# 4. Assert input_count = accepted_count + rejected_count.
# 5. Assert the exact accepted order_id set.
# 6. Assert the multi-defect row contains exactly:
#       MISSING_CUSTOMER_ID
#       NEGATIVE_NET_SALES
#    Compare reasons as a set unless order itself is contractual.
# 7. Assert the orphan row contains ORPHAN_CUSTOMER_ID.


<a id="experiment-11"></a>
# Experiment 11 — Deterministic Window Testing

Fixture:

```text
R001 | C001 | 2026-05-01 | ON
R002 | C001 | 2026-05-01 | QC
```

The dates deliberately tie.

Contract:

```text
effective_date DESC
customer_record_id DESC
```

Predict the winner before running.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Build the tied customer-history fixture.
# 2. Run phase_09.select_latest_customer_record().
# 3. Assert the exact winning customer_record_id and province.
#
# Explain why different effective dates would fail to test the secondary
# tie-breaker.


<a id="experiment-12"></a>
# Experiment 12 — Edge Cases

Required cases:

```text
empty input
numeric boundary: -0.01, 0.00, 0.01
```

For the filter boundary:

```text
minimum_net_sales = 0.00
```

Predict which keys survive.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# EMPTY INPUT
# - create empty orders/customers DataFrames with explicit schemas;
# - run validation;
# - assert 0 validated, 0 accepted, 0 rejected.
#
# NUMERIC BOUNDARY
# - build -0.01, 0.00, 0.01 COMPLETED rows;
# - call phase_09.filter_orders(... minimum_net_sales=Decimal('0.00'));
# - assert exact surviving keys.


<a id="experiment-13"></a>
# Experiment 13 — Integration Testing

Integration boundary:

```text
orders_df + customers_df
        ↓
validate_orders()
        ↓
split accepted / rejected
        ↓
transform_accepted_orders()
        ↓
build_sales_by_province()
        ↓
final assertions
```

Use:

```text
valid ON order
valid BC order
orphan order
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Build the three-order fixture and customer parent fixture.
# 2. Validate.
# 3. Split accepted/rejected.
# 4. Enrich accepted orders.
# 5. Aggregate to province grain.
#
# Assert:
# - exact province output rows;
# - input classification reconciliation;
# - accepted net_sales = output net_sales;
# - one row per province.
#
# Explain why this is integration testing rather than unit testing.


<a id="experiment-14"></a>
# Experiment 14 — Distributed Spark Testing Cost

Use one tiny DataFrame and deliberately call:

```text
count()
first()
collect()
```

Identify each as a Spark action.

Then explain when one tiny `collect()` followed by several Python assertions may be clearer than several separate Spark actions.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Create a tiny DataFrame such as spark.range(5).
# 2. Call count(), first(), and collect().
# 3. Assert the results.
# 4. Explain why each is an action.
#
# Do not infer from this exercise that collecting large production datasets is
# safe.


<a id="experiment-15"></a>
# Experiment 15 — Regression Detection

Baseline production contract:

```text
build_sales_by_province()
→ sum(net_sales)
```

Deliberate mutation:

```text
sum(net_sales)
→ avg(net_sales)
```

Your reconciliation assertion should detect the corruption.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# 1. Write deliberately_broken_sales_by_province() using avg(net_sales).
# 2. Run it on a fixture where AVG differs from SUM.
# 3. Use your reconciliation assertion.
# 4. Prove the assertion fails.
# 5. Record which business contract detected the defect.
#
# Do not modify the real Phase 9 implementation in this notebook.


<a id="applied-project"></a>
# Applied Phase 11 Project

Build a real `pytest` suite around the existing Phase 9/10 retail logic.

Suggested structure:

```text
tests/
├── conftest.py
├── test_transformations.py
├── test_validation.py
└── test_pipeline_integration.py
```

The suite should protect:

```text
schema contracts
transformation correctness
deterministic output
row population
grain
primary/composite-key uniqueness
referential integrity
accepted/rejected behavior
numerical reconciliation
edge cases
meaningful integration
```

[Back to Table of Contents](#toc)


## Applied Task Requirements

### `conftest.py`

Create:

```text
session-scoped Spark fixture
local[2]
small shuffle partition count
clean shutdown
```

### `test_transformations.py`

Protect at minimum:

```text
filter_orders()
build_sales_by_province()
add_processing_date()
select_latest_customer_record()
```

### `test_validation.py`

Protect at minimum:

```text
required-field rejection
domain/range rejection
duplicate order_id
duplicate composite inventory key
orphan customer_id
multiple rejection reasons
accepted/rejected reconciliation
```

### `test_pipeline_integration.py`

Protect:

```text
validation
→ accepted/rejected split
→ enrichment
→ aggregation
→ schema / grain / values / totals
```

### Regression requirement

Start with the suite passing.

Deliberately break one meaningful contract, for example:

```text
sum(net_sales)
→
avg(net_sales)
```

Confirm the relevant test fails.

Restore the correct implementation.

Confirm the suite passes again.

This is still practice; formal mastery is reviewed separately.

[Back to Table of Contents](#toc)


In [ ]:
# TODO:
# Use this cell for notes from the applied pytest implementation.
#
# Record:
# - files created;
# - test count;
# - important contracts protected;
# - deliberate regression introduced;
# - failing test that detected it;
# - final passing result after restoration.


<a id="cleanup"></a>
# Cleanup

[Back to Table of Contents](#toc)


In [ ]:
spark.stop()
print('Spark session stopped.')
